# Badge 1: Data Warehousing Workshop: Lesson 12: Nested Semi-Structured Data

NESTED_INGEST_JSON, FLATTEN, and the SOCIAL_MEDIA_FLOODGATES tweet data.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, which are swapped for `BADGE1_DWW_ADMIN` (ACCOUNTADMIN), `BADGE1_DWW_ROLE` (SYSADMIN) and `BADGE1_DWW_WH` (COMPUTE_WH).
- Cells marked **SQL equivalent** replace a UI-only step; **Derived solution** marks challenge-lab answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on the `@UTIL_DB.PUBLIC.BADGE1_DOWNLOADS` stage, which stands in for the learner's Downloads folder.

Prerequisite: run `dora/dora_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE1_DWW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE1_DWW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE1_DWW_WH"      # stands in for the course's COMPUTE_WH

In [ ]:
use role {{ADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
select current_role(), current_warehouse();

## Page: 🎭 Nested JSON

Video and knowledge checks only.

## Page: 🥋 Load and Query Nested Author & Book JSON Data

In [ ]:
use role {{SYSADMIN_ROLE}};
-- create an Ingestion Table for the NESTED JSON Data
create or replace table library_card_catalog.public.nested_ingest_json 
(
  raw_nested_book VARIANT
);

> **Course:** 🎯 Download the data file (json_book_author_nested.txt) and load it into NESTED_INGEST_JSON. You can use the file format you created for the non-nested JSON data.

In [ ]:
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('json_book_author_nested.txt');
list @util_db.public.my_internal_stage;

In [ ]:
-- Derived solution
use role {{ADMIN_ROLE}};
copy into library_card_catalog.public.nested_ingest_json
from @util_db.public.my_internal_stage
files = ('json_book_author_nested.txt')
file_format = (format_name = library_card_catalog.public.json_file_format);

In [ ]:
use schema library_card_catalog.public;
-- a few simple queries
select raw_nested_book
from nested_ingest_json;

select raw_nested_book:year_published
from nested_ingest_json;

select raw_nested_book:authors
from nested_ingest_json;

In [ ]:
-- use these example flatten commands to explore flattening the nested book and author data
select value:first_name
from nested_ingest_json
,lateral flatten(input => raw_nested_book:authors);

select value:first_name
from nested_ingest_json
,table(flatten(raw_nested_book:authors));

-- add a CAST command to the fields returned
SELECT value:first_name::varchar, value:last_name::varchar
from nested_ingest_json
,lateral flatten(input => raw_nested_book:authors);

-- assign new column  names to the columns using "AS"
select value:first_name::varchar as first_nm
, value:last_name::varchar as last_nm
from nested_ingest_json
,lateral flatten(input => raw_nested_book:authors);

## Page: 🎭 PATH Notation for Nested Entities

Video and knowledge checks only.

### 🤖 DORA check DWW17 goes here
The course has the learner run **DWW17** (Check number of rows, expects 5) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW17**, as a local check plus the exact DORA code.

## Page: 🎭 Some Old Twitter Tweets / 🥋 Explore the Data with an Online Parser

> **Course:** Download nutrition_tweets.json and open it in https://jsoneditoronline.org/ (DWW_167–170). Outside Snowflake; nothing to run.

> ⚠️ **Audit note:** The tweet data is from an old Twitter/X API export. The file starts with a UTF-8 byte-order mark, which Snowflake's JSON loader skips by default.

## Page: 🥋 Load and Query Nested TWEET JSON Data

> **Course:** 🎯 Create a database named SOCIAL_MEDIA_FLOODGATES. Create a table called TWEET_INGEST in its PUBLIC schema with one column named RAW_STATUS. Create a FILE FORMAT of type JSON. Write a COPY INTO that loads the tweet data; stage the file wherever you decide. You should end up with 9 rows.

In [ ]:
-- Derived solution
use role {{SYSADMIN_ROLE}};
create database SOCIAL_MEDIA_FLOODGATES;
create table social_media_floodgates.public.tweet_ingest (raw_status variant);
create file format social_media_floodgates.public.json_file_format
    type = 'JSON'
    strip_outer_array = TRUE;

In [ ]:
-- SQL equivalent of uploading the downloaded file(s) through the stage's + Files dialog
use role {{ADMIN_ROLE}};
copy files into @util_db.public.my_internal_stage
from @UTIL_DB.PUBLIC.BADGE1_DOWNLOADS
files = ('nutrition_tweets.json');
list @util_db.public.my_internal_stage;

In [ ]:
-- Derived solution
use role {{ADMIN_ROLE}};
copy into social_media_floodgates.public.tweet_ingest
from @util_db.public.my_internal_stage
files = ('nutrition_tweets.json')
file_format = (format_name = social_media_floodgates.public.json_file_format);

In [ ]:
use schema social_media_floodgates.public;
//simple select statements -- are you seeing 9 rows?
select raw_status
from tweet_ingest;

select raw_status:entities
from tweet_ingest;

select raw_status:entities:hashtags
from tweet_ingest;

//Explore looking at specific hashtags by adding bracketed numbers
//This query returns just the first hashtag in each tweet
select raw_status:entities:hashtags[0].text
from tweet_ingest;

//This version adds a WHERE clause to get rid of any tweet that 
//doesn't include any hashtags
select raw_status:entities:hashtags[0].text
from tweet_ingest
where raw_status:entities:hashtags[0].text is not null;

//Perform a simple CAST on the created_at key
//Add an ORDER BY clause to sort by the tweet's creation date
select raw_status:created_at::date
from tweet_ingest
order by raw_status:created_at::date;

## Page: 🥋 Flatten & Isolate Entities

In [ ]:
//Flatten statements can return nested entities only (and ignore the higher level objects)
select value
from tweet_ingest
,lateral flatten
(input => raw_status:entities:urls);

select value
from tweet_ingest
,table(flatten(raw_status:entities:urls));

In [ ]:
//Flatten and return just the hashtag text, CAST the text as VARCHAR
select value:text::varchar as hashtag_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:hashtags);

//Add the Tweet ID and User ID to the returned table so we could join the hashtag back to it's source tweet
select raw_status:user:name::text as user_name
,raw_status:id as tweet_id
,value:text::varchar as hashtag_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:hashtags);

### 🤖 DORA check DWW18 goes here
The course has the learner run **DWW18** (Check number of rows, expects 9) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW18**, as a local check plus the exact DORA code.

## Page: 🥋 Normalize the Hashtags

In [ ]:
create or replace view social_media_floodgates.public.urls_normalized as
(select raw_status:user:name::text as user_name
,raw_status:id as tweet_id
,value:display_url::text as url_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:urls)
);

> **Course:** 🎯 Create a view called HASHTAGS_NORMALIZED. The results of SELECT * on your view should look like DWW_172 (USER_NAME, TWEET_ID, HASHTAG_USED; 14 rows).

In [ ]:
-- Derived solution, modeled on URLS_NORMALIZED and the hashtag flatten query
create or replace view social_media_floodgates.public.hashtags_normalized as
(select raw_status:user:name::text as user_name
,raw_status:id as tweet_id
,value:text::varchar as hashtag_used
from tweet_ingest
,lateral flatten
(input => raw_status:entities:hashtags)
);

In [ ]:
select * from social_media_floodgates.public.hashtags_normalized;

In [ ]:
print('Rows:', len(check_hashtags), '(DWW_172 shows 14)')

### 🤖 DORA check DWW19 goes here
The course has the learner run **DWW19** (HASHTAGS_NORMALIZED view exists) at this point.
It is in `dora/dora_tests.ipynb`, section **DWW19**, as a local check plus the exact DORA code.

## Page: 🎉 Badge Time!

> **Course:** Check in with DORA in the YSA app: all DORA tests DWW01–DWW19 should be PASSING and VALID. Outside Snowflake; nothing to run.